## Loading The Data
We start by loading the 20 newsgroups dataset's raw text data, and split it into train and test.

In [2]:
from sklearn.datasets import fetch_20newsgroups

# Load the full training and testing dataset.
data_train = fetch_20newsgroups(subset='train', shuffle=True, random_state=42)
data_test = fetch_20newsgroups(subset='test', shuffle=True, random_state=42)

# Get the raw documents and labels.
X_train, y_train = data_train.data, data_train.target
X_test, y_test = data_test.data, data_test.target

# Target names (class labels):
target_names = data_train.target_names

print('Training data samples:', len(X_train))
print('Testing data samples:', len(X_test))
print('Number of classes:', len(target_names))

Training data samples: 11314
Testing data samples: 7532
Number of classes: 20


## Vectorizing The Data
We will vectorize the data in two ways, 1. TF-IDF 2.Count Vectorizer
Count Vectorizer use raw counts of words and TF-IDF gives a score based on importance, let's see the difference with a random document example.

In [20]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

import numpy as np
import pandas as pd

# --------------------
# 1. Fit Vectorizers
# --------------------
count_vect = CountVectorizer()
X_train_count = count_vect.fit_transform(X_train)
X_test_count = count_vect.transform(X_test)

tfidf_vect = TfidfVectorizer()
X_train_tfidf = tfidf_vect.fit_transform(X_train)
X_test_tfidf = tfidf_vect.transform(X_test)


# --------------------
# 2. Pick a Single Document
# --------------------
doc_index = 5 # can choose any document index from training set
doc_text = X_train[doc_index]

# --------------------
# 3. Extract the Vector for That Document
# --------------------
doc_count_vec = X_train_count[doc_index]
doc_tfidf_vec = X_train_tfidf[doc_index]

doc_count_arr = doc_count_vec.toarray().flatten() # Convert to dense array and flatten
doc_tfidf_arr = doc_tfidf_vec.toarray().flatten() # Convert to dense array and flatten

# Get feature names
count_features = count_vect.get_feature_names_out()
tfidf_features = tfidf_vect.get_feature_names_out()

# --------------------
# 4. Find Nonzero Elements
# --------------------
nonzero_count_idx = doc_count_arr.nonzero()[0]    # indices of nonzero tokens (Count)
nonzero_tfidf_idx = doc_tfidf_arr.nonzero()[0]    # indices of nonzero tokens (TF-IDF)


# --------------------
# 5. Combine into a Single DataFrame
# --------------------

# For a fair comparison, let's focus on words that appear at least once in this document,
# which means the union of nonzero_count_idx and nonzero_tfidf_idx.
nonzero_union = np.union1d(nonzero_count_idx, nonzero_tfidf_idx)

features = [count_features[i] for i in nonzero_union]
counts = doc_count_arr[nonzero_union]
tfidfs = doc_tfidf_arr[nonzero_union]

# 'Count' - Number of times the token(word) appears in the document
# 'TF-IDF' - TF-IDF score of the token in the document
df = pd.DataFrame({
    'token index': nonzero_union,
    'token': features,
    'count': counts,
    'tfidf': tfidfs
})

# Sort by count or tfidf (descending)
df_sorted = df.sort_values(by='count', ascending=False)
df_sorted = df_sorted.set_index('token').drop(columns=['token index']).T
# --------------------
# 6. Display the Comparison
# --------------------
print("\n=== TOKEN COUNTS vs. TF-IDF (Single Document) ===\n")
display(df_sorted)



=== TOKEN COUNTS vs. TF-IDF (Single Document) ===



token,the,of,weapons,to,destruction,and,mass,in,com,you,...,massive,mean,commonly,millions,coming,must,class,need,checks,083057
count,17.000000,17.000000,13.000000,10.00000,7.000000,7.000000,7.000000,7.000000,6.000000,6.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.00000,1.000000,1.000000
tfidf,0.120301,0.127689,0.425122,0.07338,0.295746,0.054898,0.232184,0.054306,0.073849,0.061392,...,0.039825,0.023889,0.040572,0.036714,0.029609,0.02164,0.031673,0.01917,0.043355,0.059228


# Classifiers Performance Analysis

Instead of manually transforming and training the models separately, we can define a pipeline that does it all in one go.

## Comparing Classifiers
### We will compare 4 different classifiers on the data:
- Logistic Regression
- Decision Tree
- kNN
- SVM

### Let's see an example of a document from our data

In [ ]:
print('Sample document:')
print(X_train[0])